# N023 · 字符串解析与有限状态机

**目标：** 按状态和合法转移解析含符号与边界的输入。

**先修：** N002, N006, N016。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 空白；符号；数字扫描；32位截断；状态转移；非法后缀。

**配套讲解来源：** [同名逐章意见](../../comment/023_string_parsing_fsm.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决两个"把字符串当数字读"的经典问题，它们长得像但要求完全不同。

**问题一（前缀转整数，`my_atoi`）**：给你一个字符串，请你模仿 C 语言的 atoi 函数，从左往右读：先跳过开头的一串空格，然后允许出现一个可选的正负号，然后把连续的数字部分拼成整数；一旦碰到第一个不是数字的字符就立刻停手，后面无论是什么都不看。如果数字超出了 32 位整数的范围，就"截断"到边界值。举个具体到字符的例子：输入 `'  -42'` → 输出 `-42` → 为什么？前面两个空格被跳过，读到 `-` 记下符号，接着把 `4`、`2` 拼成 42，加上符号得 −42，落在 32 位范围内，原样返回。再比如 `'4193 with words'` → 输出 `4193`，因为读到空格就停了，"with words" 全部被无视。

**问题二（完整数字校验，`is_number`）**：给你一个字符串，问它整体是不是一个"合法的十进制数"，允许小数点和科学计数法。比如 `'3.5e2'` → 输出 `True`（就是 3.5×10²）；`'1e'` → 输出 `False`，因为 e 后面必须跟数字，它光秃秃地结束了。

两个问题的分野是 “思路推导”部分 里最重要的一句话：`my_atoi` 允许在第一个非法字符处停下来（前缀解析），`is_number` 要求去掉首尾空白后的**整个串**都合法（完整校验）。同一个 `'4193 with words'`，前者的答案是 4193，后者的答案是 False——因为它有没啃完的尾巴。

## 2. 基础知识：先读懂这些词

- **前导空白（leading whitespace）**：字符串开头的一串空格。两种解析都容忍它：atoi 用循环跳过，is_number 用 `strip()` 掐头去尾。它不属于数字的一部分，只是"客气一下"。
- **可选符号（optional sign）**：数字开头的单个 `+` 或 `-`。说"可选"是因为可以没有；说"单个"是因为 `+-3` 这种连续两个符号是非法的。在科学计数法里，符号除了出现在开头，还允许紧跟在指数 e 的后面（如 `6e-1`），这是 is_number 里最绕的一条规则。
- **数字扫描（digit scanning）**：把连续的数字字符拼成一个整数的动作，核心公式是 `value = value*10 + 当前位`。就像你在纸上读数：看到 4 记住 4，看到 2 就把 4 抬到十位再加 2，得到 42。
- **32 位截断（32-bit clamping）**：32 位有符号整数只能装下 −2147483648 到 2147483647 之间的数（即 −2³¹ 到 2³¹−1）。超出范围的数不是报错，而是被"削"到最近的边界：太大的变成 2147483647，太小的变成 −2147483648。
- **有限状态机（finite state machine，FSM）**：一种"带着记忆逐字符判断"的思路——把到目前为止读过的内容压缩成少数几个开关（本章是三个布尔标志），每读一个字符就按规则拨动开关，读完后看开关的状态下结论。它省去了为每种字符串形态写一个 if 的麻烦。
- **三个状态标志（seen_digit / seen_dot / seen_exp）**：is_number 的全部记忆。"见过数字吗"（seen_digit）、"见过小数点吗"（seen_dot）、"见过指数 e 吗"（seen_exp）。小数点全场至多一个、且必须在 e 之前；e 全场至多一个、且之前必须见过数字；见到 e 之后要把 seen_digit 清零重记，因为指数部分需要自己的数字。
- **非法后缀（illegal suffix）**：合法前缀后面跟了不该有的东西。对 atoi 来说非法后缀无所谓（停了就行）；对 is_number 来说非法后缀直接判死刑——这是两个函数语义上最大的差别。
- **ord（字符转编码）**：`ord(c)` 返回字符的编码数值。`ord('7')-ord('0')` 等于 7，因为数字字符 0 到 9 的编码是连续排列的，相减正好把字符还原成数值。这是不查表把 `'7'` 变成整数 7 的标准手法。

## 3. 思路推导：从小例子开始

**`my_atoi` 的思路：**

- **Step 1：跳过前导空格。** 下标 i 从 0 出发，只要还没到头且当前字符是空格，就 i 前进一格。`'  -42'` 走完这步 i 停在 2（指向 `-`）。
- **Step 2：吃掉可选符号。** 看一眼当前字符是不是 `+` 或 `-`：是 `-` 就把 sign 记成 −1，是 `+` 或没有符号就记成 1，然后 i 前进。注意符号至多吃一个，`--3` 会在第二步后卡在第二个 `-` 上停机，返回 0。
- **Step 3：拼数字。** 只要当前字符是 0–9，就执行 `value = value*10 + 这一位数字`，i 前进。碰到任何非数字（空格、字母、小数点都算）循环自然停住。
- **Step 4：截断到 32 位范围。** 给 sign*value 穿上"上下限夹板"，超出就取边界值，返回。

**手算演示：`my_atoi('  -42')` 一格一格走**

| 步骤 | i 指向 | 动作 | 状态 |
|---|---|---|---|
| 跳空格 | s[0]=' ' | i→1 | |
| 跳空格 | s[1]=' ' | i→2 | |
| 读符号 | s[2]='-' | sign=−1，i→3 | |
| 拼数字 | s[3]='4' | value=0*10+4=4，i→4 | |
| 拼数字 | s[4]='2' | value=4*10+2=42，i→5 | |
| 停机 | i=5=长度 | 退出循环 | 返回 max(−2³¹, min(2³¹−1, −42)) = −42 |

再看截断的例子：`'91283472332'` 拼出 91283472332，超过上限 2147483647，被夹板压成 2147483647 返回；`'-91283472332'` 拼出 −91283472332，低于下限 −2147483648，被抬到 −2147483648 返回。

**`is_number` 的思路：**

- **Step 1：strip 掐掉首尾空白。** 注意只掐首尾，中间的空格（如 `'4193 with words'`）掐不掉，会在 Step 3 里被判非法。
- **Step 2：空串直接判负。** 掐完什么都没剩（比如输入是 `'   '` 或 `'+'` 之类），说明连一个数字都没有，返回 False。
- **Step 3：带着三个开关逐字符过。** 每个字符按类别处理：数字就点亮 seen_digit；小数点要求"以前没见过点、也没见过 e"，然后点亮 seen_dot；e 要求"以前没见过 e、而且之前见过数字"，点亮 seen_exp 的同时**把 seen_digit 清零**（因为指数部分要重新攒数字）；符号只允许出现在第 0 位或紧跟 e 之后；其余任何字符直接判负。
- **Step 4：用开关下结论。** 全部读完且中间没被判负时，答案就是 seen_digit——它为真表示"最后一段（指数后的部分，如果没有 e 就是整体）确实有数字"。

**手算演示：`is_number('3.5e2')` 的开关变化（对应 “运行示例”部分 表格）**

| i | 字符 | 命中分支 | 检查 | 开关变化 |
|---|---|---|---|---|
| 0 | 3 | 数字 | — | seen_digit=True |
| 1 | . | 小数点 | 没见过点、没见过 e，通过 | seen_dot=True |
| 2 | 5 | 数字 | — | seen_digit=True |
| 3 | e | 指数 | 没见过 e、之前有数字，通过 | seen_exp=True，**seen_digit=False** |
| 4 | 2 | 数字 | — | seen_digit=True |

读完返回 seen_digit=True，所以 `'3.5e2'` 是合法数字。对照组 `'1e'`：读到 1 时 seen_digit=True，读到 e 时 seen_exp=True 且 seen_digit 被清零，字符串到此结束，返回 seen_digit=False——e 后面没有数字，判负。“运行示例”部分 表格里 `'  -42'` 的 is_number 是 True、`'4193 with words'` 是 False（中间空格命中"其余字符"分支）、`'.8'` 是 True（小数点前可以没有数字，只要后面有）、`'+'` 是 False（从头到尾没见过数字），你可以逐条对照上面的规则自己复核。

## 4. 核心代码：my_atoi

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def my_atoi(s):
    i = 0
    n = len(s)
    while i < n and s[i] == ' ':
        i += 1
    sign = 1
    if i < n and s[i] in '+-':
        sign = -1 if s[i] == '-' else 1
        i += 1
    value = 0
    while i < n and '0' <= s[i] <= '9':
        value = value * 10 + ord(s[i]) - ord('0')
        i += 1
    return max(-(1 << 31), min((1 << 31) - 1, sign * value))
```

### 逐段读懂代码

### 函数 1：`my_atoi`（前缀转整数）

先贴 notebook “分段实现”部分 里的真实代码：

```python
def my_atoi(s):
    i = 0
    n = len(s)
    while i < n and s[i] == ' ':
        i += 1
    sign = 1
    if i < n and s[i] in '+-':
        sign = -1 if s[i] == '-' else 1
        i += 1
    value = 0
    while i < n and '0' <= s[i] <= '9':
        value = value * 10 + ord(s[i]) - ord('0')
        i += 1
    return max(-(1 << 31), min((1 << 31) - 1, sign * value))
```

- **第 2 行 `i=0; n=len(s)`**：i 是当前读到的下标（游标），n 是串长缓存，避免循环里反复调 len。分号连接的两条赋值就是普通的两行写紧凑了。
- **第 3 行 `while i<n and s[i]==' ': i+=1`**：跳前导空格。两个条件用 and 串联：`i<n` 先保证下标没越界（Python 的短路求值保证越界时不会真的去取 s[i]），`s[i]==' '` 保证当前确实是空格。整串都是空格时 i 会安全地停在 n，后面每一步也都有 `i<n` 把门。
- **第 4 行 `sign=1`**：符号默认为正。这样没有符号字符时不用特殊处理，最后一乘就是原值。
- **第 5–6 行 `if i<n and s[i] in '+-': sign=-1 if s[i]=='-' else 1; i+=1`**：吃可选符号。`s[i] in '+-'` 是"当前字符是加号或减号"的紧凑写法（成员检查，对单字符字符串同样适用）。`sign=-1 if s[i]=='-' else 1` 是三元表达式：是减号就 −1，否则（必是加号）就 1。注意这整段只在有符号时执行一次，天然保证"至多一个符号"。
- **第 7–9 行拼数字**：`'0'<=s[i]<='9'` 是 Python 特有的链式比较，等价于 `'0'<=s[i] and s[i]<='9'`，用来判断"是 ASCII 数字字符"。核心一句是 `value=value*10+ord(s[i])-ord('0')`：先让旧的值整体左移一位（乘 10），再补上新数字。以 `'42'` 为例：第一轮 value=0*10+4=4，第二轮 value=4*10+2=42。碰到非数字字符条件失效，循环停下——这就是"停止于第一个非法字符"。本章限定 ASCII 数字，全角数字等不在考虑范围内。
- **第 10 行 `return max(-(1<<31),min((1<<31)-1,sign*value))`**：32 位截断的三明治写法。`1<<31` 是二进制 1 左移 31 位，即 2147483648；`(1<<31)-1` 是上限 2147483647，`-(1<<31)` 是下限 −2147483648。先做 `min(上限, sign*value)`：值若超过上限就被压到上限；再做 `max(下限, ...)`：若低于下限就被抬到下限。两层夹出来的数保证落在区间内。Python 的整数本来不会溢出，所以可以放心先拼完再夹；“正确性与复杂度”部分 也说明这是教学直写，工程上超界可提前饱和以省掉大数计算的开销。

### 函数 2：`is_number`（完整数字校验）

（对应程序见下方分段实现与完整代码。）

- **第 2 行 `s=s.strip()`**：掐掉首尾空白（空格、换行都算），中间的留着。这行执行后 s 变成了待判定的净内容。
- **第 3 行 `if not s: return False`**：掐完就空了，说明一个有效字符都不剩，必不可能是数字。空串、纯空格串都被它挡住。
- **第 4 行**：三个状态开关全部初始化为 False——这就是有限状态机的"初始状态"。
- **第 5 行 `for i,c in enumerate(s)`**：同时拿下标 i 和字符 c。i 在后面判断"符号是否在开头/紧跟 e"时要用。
- **第 6 行数字分支**：任何数字字符都点亮 seen_digit，无附加条件——数字在任何位置（小数前、小数后、指数前、指数后）都是合法的。
- **第 7–8 行符号分支**：符号只允许两种位置——第 0 位（`i==0`），或者前一个字符是 e/E（`s[i-1] in 'eE'`）。条件 `if i!=0 and s[i-1] not in 'eE': return False` 读作"既不在开头、前面又不是 e，就判负"。`'+'`、`'-0.1'`、`'+6e-1'` 里的符号分别落在两种合法位置上。
- **第 9–11 行小数点分支**：`if seen_dot or seen_exp: return False` 挡两种情况——已经见过小数点（第二个点，如 `'1.2.3'`），或者已经进入指数部分（`'1e2.3'` 里的点非法，指数必须是整数）。通过检查后点亮 seen_dot。注意点的前后都可以没有数字（`.8`、`3.` 都合法），所以这里不碰 seen_digit。
- **第 12–14 行指数分支**：两道前置检查——`seen_exp` 为真说明这是第二个 e，判负；`not seen_digit` 说明 e 之前一个数字都没有（如 `'e3'`、`'.'` 接 `'e'`），也判负。通过后点亮 seen_exp，并**把 seen_digit 清零**。清零是全函数最精妙的一步：seen_digit 的语义从"主体有数字"切换成"指数部分有数字"，这样最后一行的返回值自动承担双重职责——没出现 e 时它问"主体有数字吗"，出现 e 后它问"指数后有数字吗"。`'1e'`、`'1e+'`、`'6e-'` 全是靠这次清零后被末尾检查抓住的。
- **第 15 行 `else: return False`**：任何不属于以上四类的字符（字母、空格、下划线……）直接判负，这正是"整个串都要合法"的体现。
- **第 16 行 `return seen_digit`**：循环没中途判负的话，seen_digit 的最终值就是答案：为真表示最后一段里有数字，为假表示收尾在数字到来之前。

## 5. 分段实现：按顺序运行

**本章接口：** `my_atoi(s)`、`is_number(s)`

### my_atoi

In [1]:
def my_atoi(s):
    i = 0
    n = len(s)
    while i < n and s[i] == ' ':
        i += 1
    sign = 1
    if i < n and s[i] in '+-':
        sign = -1 if s[i] == '-' else 1
        i += 1
    value = 0
    while i < n and '0' <= s[i] <= '9':
        value = value * 10 + ord(s[i]) - ord('0')
        i += 1
    return max(-(1 << 31), min((1 << 31) - 1, sign * value))

### is_number

In [2]:
def is_number(s):
    s = s.strip()
    if not s:
        return False
    seen_digit = False
    seen_dot = False
    seen_exp = False
    for i, c in enumerate(s):
        if '0' <= c <= '9':
            seen_digit = True
        elif c in '+-':
            if i != 0 and s[i - 1] not in 'eE':
                return False
        elif c == '.':
            if seen_dot or seen_exp:
                return False
            seen_dot = True
        elif c in 'eE':
            if seen_exp or not seen_digit:
                return False
            seen_exp = True
            seen_digit = False
        else:
            return False
    return seen_digit

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['输入','atoi前缀解析','完整数字判定'],[(s,my_atoi(s),is_number(s)) for s in ['  -42','4193 with words','3.5e2','1e','+','.8','9e-3']])

输入,atoi前缀解析,完整数字判定
-42,-42,True
4193 with words,4193,False
3.5e2,3,True
1e,1,False
+,0,False
.8,0,True
9e-3,9,True


## 7. 正确性、适用条件与复杂度

指数前必须已有数字，读到指数后重置seen_digit，最终要求指数部分有数字；小数点只能在指数之前出现一次，符号只允许开头或紧跟指数。

**代价：** O(n) 字符扫描；atoi逐步大整数计算存在位代价，可在超界确定后提前饱和以限制位数。本实现为教学直写。

### 展开理解

**`my_atoi` 为什么是对的？** 四个阶段各管一段，且顺序与题目规则一一对应：跳空格的循环只消费开头的空格，遇到第一个非空格就让位；符号阶段最多吃一个字符且只认 `+`/`-`；数字循环的执行条件本身就是"当前是数字"，所以它拼出的 value 恰好是"从符号之后的第一个数字起、到第一个非数字字符为止"的那段前缀的值，之后的内容一概没被消费；最后的夹板把任何越界值映射到最近的边界，正是题目规定的截断行为。四个阶段拼起来覆盖了"空白、符号、数字、截断"全部规则，又互不越界，所以结果与 atoi 的定义吻合。

**`is_number` 为什么是对的？** 它的判断标准可以整理成五条规则：一、指数之前必须已经出现过数字；二、见到 e 之后重置 seen_digit，并在最后要求指数部分真的有数字；三、小数点至多一个且必须在 e 之前；四、符号只允许出现在开头或紧跟 e；五、除了数字、点、e、符号以外的字符一律非法。代码的五个分支恰好一一实现这五条：任何违反都会当场 `return False`，而全部通过时，末尾的 seen_digit 为真恰好等价于"整体（或指数部分）有数字"。因为返回 True 当且仅当五条规则全部满足，而这个规则集正是"合法十进制数"的定义，所以判断与定义一致。

**复杂度是多少？** 两个函数都只从头到尾扫一遍字符串，每个字符只做常数次比较和开关操作，时间 O(n)。具体到数字：一个 200 字符的输入大约就是两百来次分支判断，微秒量级，毫无压力。空间上 my_atoi 只用 i、sign、value 等常数个变量，O(1)；is_number 也只有三个布尔开关，O(1)。“正确性与复杂度”部分 补充的一点值得知道：Python 整数不限位数，`my_atoi` 拼一个超长数字串时，value 会越长越大，每乘 10 的代价随之增长，严格算有额外的位代价；工程实现会在确定必然越界后提前饱和返回边界值，把有效位数限制在十位左右。本章为了教学清晰选择了直写。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言分两组看：

- `assert my_atoi('')==0 and my_atoi('  -42')==-42`：边界值加正常值。空串没有任何数字可拼，value 保持初始 0，测的是"空输入安全返回 0"；`'  -42'` 测的是空格、符号、数字三阶段的完整接力。
- `assert my_atoi('4193 with words')==4193`：测前缀语义——数字后面跟了一坨别的东西，函数必须在空格处停住，只交出 4193，而不是报错也不是试图继续读。
- `assert my_atoi('91283472332')==(1<<31)-1`：正向溢出截断。拼出的 91283472332 超过上限 2147483647，测的是夹板的上半层把它压到上限。
- `assert my_atoi('-91283472332')==-(1<<31)`：负向溢出截断。−91283472332 低于下限 −2147483648，测的是夹板的下半层把它抬到下限。注意下限的绝对值比上限大 1，这条专门确认没有写错边界。
- `for s in ['1e','e3','+','.','1e+','1.2.3','6e-']: assert not is_number(s)`：七个非法样本各打一条规则的脸——`'1e'` 打"指数后要有数字"，`'e3'` 打"指数前要有数字"，`'+'` 和 `'.'` 打"整体必须见过数字"，`'1e+'` 打"e 后的符号后面仍需数字"，`'1.2.3'` 打"小数点至多一个"，`'6e-'` 同样打"指数后有符号无数字"。
- `for s in ['2','-0.1','2e10','.8','3.','+6e-1']: assert is_number(s)`：六个合法样本各验证一种容易被误判为非法的形态——`'-0.1'` 是负小数，`'.8'` 是无整数部分的小数，`'3.'` 是无小数部分的小数，`'+6e-1'` 是"开头正号 + 指数负号"双符号组合。这组测的是规则的宽容边界，防止把合法写法错杀。

两条 for 循环跑完才轮到 `print('N023: 所有本章断言通过')`，所以看到这行输出就代表 4 条 atoi 断言和 13 个 is_number 样本全部过关。

In [4]:
assert my_atoi('') == 0 and my_atoi('  -42') == -42

assert my_atoi('4193 with words') == 4193

assert my_atoi('91283472332') == (1 << 31) - 1

assert my_atoi('-91283472332') == -(1 << 31)

for s in ['1e', 'e3', '+', '.', '1e+', '1.2.3', '6e-']:
    assert not is_number(s)

for s in ['2', '-0.1', '2e10', '.8', '3.', '+6e-1']:
    assert is_number(s)

print('N023: 所有本章断言通过')

N023: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 区分完整数字校验和前缀转整数。

**练习 2：** 为指数符号设计状态。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（区分完整校验与前缀转整数）**：方向是找一组"同一个字符串、两个函数给出不同结论"的样本，把分界线说清楚。比如 `'3.5e2'`：is_number 说 True（整体是合法数字），而 my_atoi 只交出前缀 3，因为扫描在点号处就停了；再如 `'1e'`：my_atoi 返回 1，is_number 返回 False。建议你列表对比三五个样本，列明输入、两函数各自的输出、以及"为什么不同"（前缀允许停止于第一个非法字符，完整校验要求整串合法）。边界别忘了空串和纯空格串——前者两函数分别得 0 和 False。
- **练习 2（为指数符号设计状态）**：方向是把 `+6e-1` 里 e 后面那个 `-` 的合法性显式地画成状态。提示：把"紧跟上一个字符是 e/E"这个信息独立成一个布尔开关（比如 seen_exp_just_now，读到 e 时点亮、处理下一个字符后熄灭），符号分支的条件 `i!=0 and s[i-1] not in 'eE'` 就能改写成对开关的检查。手算示例可以用 `'1e+5'`（合法）和 `'1e++5'`（第二个 + 时开关已灭，判负），列一张"字符—开关变化—判定"的表，再用本章 `is_number` 验证你的状态设计给出同样结论。

两道练习的本质都是把"隐式的位置信息"（上一个字符是什么）翻译成"显式的状态变量"，这正是有限状态机这个名称的含义。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def my_atoi(s):
    i = 0
    n = len(s)
    while i < n and s[i] == ' ':
        i += 1
    sign = 1
    if i < n and s[i] in '+-':
        sign = -1 if s[i] == '-' else 1
        i += 1
    value = 0
    while i < n and '0' <= s[i] <= '9':
        value = value * 10 + ord(s[i]) - ord('0')
        i += 1
    return max(-(1 << 31), min((1 << 31) - 1, sign * value))

def is_number(s):
    s = s.strip()
    if not s:
        return False
    seen_digit = False
    seen_dot = False
    seen_exp = False
    for i, c in enumerate(s):
        if '0' <= c <= '9':
            seen_digit = True
        elif c in '+-':
            if i != 0 and s[i - 1] not in 'eE':
                return False
        elif c == '.':
            if seen_dot or seen_exp:
                return False
            seen_dot = True
        elif c in 'eE':
            if seen_exp or not seen_digit:
                return False
            seen_exp = True
            seen_digit = False
        else:
            return False
    return seen_digit

# 示例与回归测试
assert my_atoi('') == 0 and my_atoi('  -42') == -42

assert my_atoi('4193 with words') == 4193

assert my_atoi('91283472332') == (1 << 31) - 1

assert my_atoi('-91283472332') == -(1 << 31)

for s in ['1e', 'e3', '+', '.', '1e+', '1.2.3', '6e-']:
    assert not is_number(s)

for s in ['2', '-0.1', '2e10', '.8', '3.', '+6e-1']:
    assert is_number(s)

print('N023: 所有本章断言通过')

N023: 所有本章断言通过


## 11. 代表题与迁移

- **8. String to Integer (atoi)（字符串转整数）**：这题就是 `my_atoi` 本尊，练的是"多阶段顺序解析 + 越界截断"——空格、符号、数字、夹板四步一步不能乱，溢出边界的正负不对称（−2³¹ 与 2³¹−1）是常见失分点。
- **65. Valid Number（有效数字）**：这题是 `is_number` 的完整版（extension），练的是用状态机处理"主体、小数点、指数、符号位置"交织的合法性问题；本章的三开关实现是它的教学简化版，做原题时可以升级成显式的状态转移表。

做完这两题，你可以自测：不看代码，说出 is_number 里 seen_digit 在读到 e 时为什么要清零、不清零会漏判哪个样本（`'1e'`），能说顺就说明状态机的账你算明白了。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。